In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI

# .env を読み込む
load_dotenv()

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_PROJECT"] = "agent-book"

In [2]:
from langchain_community.document_loaders import GitLoader

def file_filter(p: str) -> bool:
    p = p.lower()
    return p.endswith(".md") or p.endswith(".mdx")

loader = GitLoader(
    clone_url="https://github.com/langchain-ai/langchain",
    repo_path="./langchain",
    branch="master",
    file_filter=file_filter,
)

documents = loader.load()
print("documents:", len(documents))
print("sample:", documents[0].metadata.get("source"), documents[0].page_content[:200])

documents: 28
sample: AGENTS.md # Global development guidelines for the LangChain monorepo

This document provides context to understand the LangChain Python project and assist with development.

## Project architecture and context



In [3]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
db = Chroma.from_documents(documents, embeddings)

In [4]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI

prompt = ChatPromptTemplate.from_template('''\
以下の文脈だけを踏まえて質問に回答してください。

文脈："""
{context}
"""

質問：{question}
''')

model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

retriever = db.as_retriever()

chain = {
    "question": RunnablePassthrough(),
    "context": retriever,
} | prompt | model | StrOutputParser()

chain.invoke("LangChainの概要を教えて")

'LangChainは、LLM（大規模言語モデル）を活用したエージェントやアプリケーションを構築するためのフレームワークです。開発者が10行未満のコードでOpenAIやAnthropic、Googleなどのモデルに接続できるように設計されており、エージェントアーキテクチャやモデル統合を提供して、迅速に開発を開始できるようにします。\n\nLangChainを使用することで、エージェントや自律的なアプリケーションを迅速に構築でき、複雑なタスクを処理するための低レベルのエージェントオーケストレーションフレームワークであるLangGraphも利用可能です。LangChainは、リアルタイムデータの拡張、モデルの相互運用性、迅速なプロトタイピング、プロダクション向けの機能を備えており、開発者がアプリケーションを効率的に構築できるようサポートします。\n\nまた、LangChainはオープンソースプロジェクトであり、コミュニティからの貢献を歓迎しており、豊富なドキュメントやリソースも提供されています。'

In [5]:
hypothetical_prompt = ChatPromptTemplate.from_template("""\
次の質問に回答する一文を書いてください。

質問：{question}
""")

hypothetical_chain = hypothetical_prompt | model | StrOutputParser()

In [6]:
from pydantic import BaseModel, Field

class QueryGenerationOutput(BaseModel):
    queries: list[str] = Field(..., description="検索クエリのリスト")

query_generation_prompt = ChatPromptTemplate.from_template("""\
質問に対してベクターデータベースから関連文書を検索するために、
 3つの異なる検索クエリを生成してください。
 距離ベースの類似性検索の限界を克服するために、
 ユーザーの質問に対して複数の視点を提供することが目標です。

質問：{question}
""")

query_generation_chain = (
    query_generation_prompt
    | model.with_structured_output(QueryGenerationOutput)
    | (lambda x: x.queries)
)

In [7]:
multi_query_rag_chain = {
    "question": RunnablePassthrough(),
    "context": query_generation_chain | retriever.map(),
} | prompt | model | StrOutputParser()

multi_query_rag_chain.invoke("LangChainの概要を教えて")

'LangChainは、LLM（大規模言語モデル）を活用したエージェントやアプリケーションを構築するためのフレームワークです。開発者が迅速にエージェントや自律的なアプリケーションを構築できるように、10行未満のコードでOpenAI、Anthropic、Googleなどのモデルに接続できる機能を提供します。\n\nLangChainは、事前に構築されたエージェントアーキテクチャやモデル統合を提供し、LLMをエージェントやアプリケーションにシームレスに組み込む手助けをします。また、LangGraphという低レベルのエージェントオーケストレーションフレームワークを使用することで、より高度なニーズに対応することも可能です。\n\n主な特徴としては、以下の点が挙げられます：\n\n- **モジュール性**: 各コンポーネントが独立しており、特定のモデルプロバイダーに依存しない設計。\n- **迅速なプロトタイピング**: モジュールベースのアーキテクチャにより、アプリケーションの迅速な構築と反復が可能。\n- **生産準備が整った機能**: モニタリングやデバッグのための統合が組み込まれており、信頼性の高いアプリケーションを展開できる。\n- **活発なコミュニティ**: 継続的な改善や最新のAI開発に関する情報を得られるオープンソースコミュニティ。\n\nLangChainは、開発者がLLMアプリケーションを構築するための標準インターフェースを提供し、さまざまなデータソースやシステムと簡単に接続できるように設計されています。'

In [11]:
from langchain_core.documents import Document

def reciprocal_rank_fusion(
    retriever_outputs: list[list[Document]],
    k: int = 60,
) -> list[str]:
    # 各ドキュメントのコンテンツ（文字列）とそのスコアの対応を保持する辞書を準備
    content_score_mapping = {}
    # 検索クエリごとにループ
    for docs in retriever_outputs:
        # 検索結果のドキュメントごとにループ
        for rank, doc in enumerate(docs):
            content = doc.page_content

            # 初めて登場したコンテンツの場合はスコアを0で初期化
            if content not in content_score_mapping:
                content_score_mapping[content] = 0

            # (1 / 順位 + k))のスコアを加算
            content_score_mapping[content] += 1 / (rank + k)

        # スコアの大きい順にソート
        ranked = sorted(content_score_mapping.items(), key=lambda x: x[1], reverse = True)
        return [content for content, _ in ranked]

In [13]:
rag_fusion_chain = {
    "question": RunnablePassthrough(),
    "context": query_generation_chain | retriever.map() | reciprocal_rank_fusion,
} | prompt | model | StrOutputParser()

rag_fusion_chain.invoke("LangGraphの概要を教えて")

'LangGraphは、LangChainの低レベルエージェントオーケストレーションフレームワークであり、複雑なタスクを処理するための信頼性の高いエージェントワークフローを構築するために使用されます。LangGraphを利用することで、決定論的なワークフローとエージェント的なワークフローを組み合わせた高度なカスタマイズが可能になり、制御されたレイテンシを維持しながら、エージェントの実行を持続可能にすることができます。基本的なLangChainエージェントの使用にはLangGraphの知識は必要ありませんが、より高度なニーズがある場合にはLangGraphを活用することが推奨されています。'

In [17]:
from langchain_community.retrievers import TavilySearchAPIRetriever

langchain_document_retriever = retriever.with_config(
    {"run_name": "langchain_document_retriever"}
)

web_retriever = TavilySearchAPIRetriever(k=3).with_config(
    {"run_name": "web_retriever"}
)

In [18]:
from enum import Enum

class Route(str, Enum):
    langchain_document = "langchain_document"
    web = "web"

class RouteOutput(BaseModel):
    route: Route

route_prompt = ChatPromptTemplate.from_template("""\
質問に回答するために適切なRetrieverを選択してください。

質問：{question}
""")

route_chain = (
    route_prompt
    | model.with_structured_output(RouteOutput)
    | (lambda x: x.route)
)

In [20]:
from typing import Any

def routed_retriever(inp: dict[str, Any]) -> list[Document]:
    question = inp["question"]
    route = inp["route"]

    if route == Route.langchain_document:
        return langchain_document_retriever.invoke(question)
    elif route == Route.web:
        return web_retriever.invoke(question)

    raise ValueError(f"Unknown retriever: {retriever}")

route_rag_chain = (
    {
        "question": RunnablePassthrough(),
        "route": route_chain,
    }
    | RunnablePassthrough.assign(context=routed_retriever)
    | prompt | model | StrOutputParser()
)

In [21]:
route_rag_chain.invoke("LangChainの概要を教えて")

'LangChainは、LLM（大規模言語モデル）を活用したエージェントやアプリケーションを構築するためのフレームワークです。開発者が簡単にエージェントを作成し、OpenAIやAnthropic、Googleなどのモデルと接続できるように設計されています。LangChainを使用することで、10行未満のコードでアプリケーションを迅速に構築でき、エージェントのアーキテクチャやモデル統合が提供されます。\n\nLangChainは、エージェントの構築を簡素化し、将来の技術の進化に対応できるように設計されています。基本的な使用法にはLangGraphという低レベルのエージェントオーケストレーションフレームワークが利用されており、より高度なニーズに応じたカスタマイズや制御が可能です。\n\nまた、LangChainは、リアルタイムデータの拡張、モデルの相互運用性、迅速なプロトタイピング、プロダクション向けの機能を提供し、開発者が信頼性の高いアプリケーションを構築できるようサポートします。コミュニティやエコシステムも活発で、さまざまな統合やテンプレートが利用可能です。'

In [22]:
route_rag_chain.invoke("東京の今日の天気は？")

'東京の今日の天気は、3月21日（土）で曇時々晴れ、最高気温は16℃、最低気温は5℃です。降水確率は0％で、風は南の風から北の風に変わります。'